# Mission 06 — Squadron Notebook (`ps06_solution.ipynb`)
## The Autonomous Warp State Machine & Deliberation Loop
*USSE Noosphere Bridge Crew · Sector `SEC-06-CAR` (Eta Carinae & Homunculus Nebula) · Subsystem 06 (+7.1% → 42.6% total integrity)*

AURA has fallen back to System 1 reflexes. This notebook restores System 2: a deterministic, halt-able state machine, a validated tool protocol, an MDP return evaluator, and a cycle-detecting circuit breaker, all in plain Python (standard library only).

| Part | Role | Officer | Implementation file |
|---|---|---|---|
| 1 | Systems Architect (Lead Engineer) | Abdul Jalil Memon | `01_Agent_State_Machine.py` |
| 2 | Communications & Protocol Officer | Fatima Siddiqa | `02_Tool_Protocol_Dispatcher.py` |
| 3 | Data & Payload Specialist | Abubaker Attique | `03_MDP_Trajectory_Logger.py`, `03_Infinite_Loop_Sentinel.py` |
| 4 | Security & Ethics Sentinel | Syeda Manahil Javed | `04_System2_Warp_Pilot.py` |

**How this notebook is organised.** Each part has a short demonstration of its own, Part 4 wires all four together, and the **Master Test Suite** at the bottom holds the four `assert` suites the briefing requires. Run it from the `mission-06/` folder (or from the repository root); the cell below loads the five `.py` files that sit next to it.

> **Role/file mapping.** The briefing gives cycle detection to the Security & Ethics Sentinel, while the mission README and the starter files give the sentinel to the Data & Payload Specialist. Both are implemented (`03_Infinite_Loop_Sentinel.py`, used by Part 4), so nothing is missing; only the labels differ.

In [1]:
import importlib.util, json, logging, math, pathlib, sys

HERE = pathlib.Path.cwd()
if not (HERE / "01_Agent_State_Machine.py").exists() and (HERE / "mission-06").is_dir():
    HERE = HERE / "mission-06"          # notebook opened from the repository root

def load(module_name, filename):
    spec = importlib.util.spec_from_file_location(module_name, HERE / filename)
    mod = importlib.util.module_from_spec(spec)
    sys.modules[module_name] = mod       # registered before exec: the Part 4 dataclasses need it
    spec.loader.exec_module(mod)
    return mod

agent_mod    = load("ps06_agent",    "01_Agent_State_Machine.py")
protocol_mod = load("ps06_protocol", "02_Tool_Protocol_Dispatcher.py")
mdp_mod      = load("ps06_mdp",      "03_MDP_Trajectory_Logger.py")
sentinel_mod = load("ps06_sentinel", "03_Infinite_Loop_Sentinel.py")
pilot_mod    = load("ps06_pilot",    "04_System2_Warp_Pilot.py")

Agent, AgentState = agent_mod.Agent, agent_mod.AgentState
print("python", sys.version.split()[0], "| loaded from", HERE.name)
for m in (agent_mod, protocol_mod, mdp_mod, sentinel_mod, pilot_mod):
    print("  ", m.__name__, "<-", pathlib.Path(m.__file__).name)

python 3.12.3 | loaded from mission-06
   ps06_agent <- 01_Agent_State_Machine.py
   ps06_protocol <- 02_Tool_Protocol_Dispatcher.py
   ps06_mdp <- 03_MDP_Trajectory_Logger.py
   ps06_sentinel <- 03_Infinite_Loop_Sentinel.py
   ps06_pilot <- 04_System2_Warp_Pilot.py


---
## Part 1 — Systems Architect: Native Python Agent State Machine Core

**Design.** `AgentState` is an `Enum` with exactly the seven required states. `Agent` is a standard `@dataclass`. The graph lives in one table (`TRANSITIONS`), so it can be printed and tested, and `Agent.step()` performs **exactly one** transition per call.

$$\text{IDLE}\to\text{PERCEIVING}\to\text{DELIBERATING}\to\text{ACTING}\to\text{OBSERVING}\to\text{REFLECTING}\to\begin{cases}\text{DELIBERATING} & \text{loop}\\ \text{HALTED} & \text{terminal}\end{cases}$$

`REFLECTING` is the only state with two successors; it goes to `HALTED` when `mission_completed` is set, otherwise back to `DELIBERATING`.

**Step budget ($N\le 20$).** `max_steps` is validated to be an integer in $1..20$ (the budget cannot be configured away). After $N$ transitions the next `step()` call forces `HALTED` instead of transitioning, so a non-terminating run is stopped on call $N+1$. `HALTED` is terminal: further `step()` calls change nothing and use no budget.

In [2]:
print("Transition graph (from TRANSITIONS):")
for src, dsts in agent_mod.TRANSITIONS.items():
    print(f"  {src.name:<13} -> {' | '.join(d.name for d in dsts) or '(terminal)'}")

print("\nAgent that never completes its mission, budget N = 10 (12 calls attempted):")
pilot = Agent(max_steps=10)
for _ in range(12):
    if pilot.is_halted:
        break
    pilot.step()
print(f"-> state={pilot.state.name}, step_count={pilot.step_count}, halt_reason={pilot.halt_reason!r}")

print("\nAgent whose mission completes after one deliberation cycle:")
done = Agent()
for _ in range(5):
    done.step("radiation spike at sector 7" if done.state == AgentState.IDLE else None)
done.mission_completed = True
done.step()
print(f"-> state={done.state.name}, step_count={done.step_count}, halt_reason={done.halt_reason!r}")
print("   memory:", done.memory)

Transition graph (from TRANSITIONS):
  IDLE          -> PERCEIVING
  PERCEIVING    -> DELIBERATING
  DELIBERATING  -> ACTING
  ACTING        -> OBSERVING
  OBSERVING     -> REFLECTING
  REFLECTING    -> DELIBERATING | HALTED
  HALTED        -> (terminal)

Agent that never completes its mission, budget N = 10 (12 calls attempted):
Step 01: Transitioned from IDLE -> PERCEIVING
Step 02: Transitioned from PERCEIVING -> DELIBERATING
Step 03: Transitioned from DELIBERATING -> ACTING
Step 04: Transitioned from ACTING -> OBSERVING
Step 05: Transitioned from OBSERVING -> REFLECTING
Step 06: Transitioned from REFLECTING -> DELIBERATING
Step 07: Transitioned from DELIBERATING -> ACTING
Step 08: Transitioned from ACTING -> OBSERVING
Step 09: Transitioned from OBSERVING -> REFLECTING
Step 10: Transitioned from REFLECTING -> DELIBERATING
[SECURITY] Maximum step budget (10) exceeded! Forcing state -> HALTED.
-> state=HALTED, step_count=10, halt_reason='step budget of 10 exceeded'

Agent whose mission

---
## Part 2 — Communications & Protocol Officer: Tool Protocol & Action Dispatcher

`ActionDispatcher.dispatch(tool_name, arguments)` runs three checks before anything touches a ship system: the tool must exist, the payload must be a dict (or a JSON string that decodes to one), and the arguments must satisfy the tool's JSON-Schema contract (required keys, types, no extras; `bool` and `NaN`/`inf` are rejected for numbers). The whole call sits inside an error boundary, so a failure comes back as the string `Tool Execution Error: ...` that is fed to the next deliberation cycle, and the Python process never crashes.

In [3]:
d = protocol_mod.ActionDispatcher()
calls = [
    ("navigate_warp_vector",  {"x": -0.85, "y": 0.52, "z": 0.05}),
    ("adjust_plasma_shields", {"intensity": 85}),
    ("query_sensor_array",    {"target": "escape_corridor"}),
    ("adjust_plasma_shields", {"intensity": -10}),            # tool raises ValueError -> error boundary
    ("adjust_plasma_shields", {"intensity": "max"}),          # schema: wrong type
    ("navigate_warp_vector",  {"x": 1.0}),                    # schema: missing arguments
    ("self_destruct",         {"confirm": True}),             # unknown tool
    ("adjust_plasma_shields", '{"intensity": 85'),            # malformed JSON text
    ("query_sensor_array",    None),                          # not a dict
]
for tool, payload in calls:
    print(f"{tool}({payload!r})\n    -> {d.dispatch(tool, payload)}")

navigate_warp_vector({'x': -0.85, 'y': 0.52, 'z': 0.05})
    -> Observation: Warp navigational coordinates successfully aligned to vector [-0.85, 0.52, 0.05].
adjust_plasma_shields({'intensity': 85})
    -> Observation: Plasma shield harmonics adjusted to 85% capacity.
query_sensor_array({'target': 'escape_corridor'})
    -> Observation: Clear gravitational corridor detected along vector [-0.85, 0.52, 0.05].
adjust_plasma_shields({'intensity': -10})
    -> Tool Execution Error: ValueError -- Shield intensity -10% is out of bounds! Valid range: [0, 100].
adjust_plasma_shields({'intensity': 'max'})
    -> Tool Execution Error: Schema validation failed for 'adjust_plasma_shields' -- Argument 'intensity' expects type 'integer', got 'str'.
navigate_warp_vector({'x': 1.0})
    -> Tool Execution Error: Schema validation failed for 'navigate_warp_vector' -- Missing required argument 'y'.; Missing required argument 'z'.
self_destruct({'confirm': True})
    -> Tool Execution Error: Unknown tool 

---
## Part 3 — Data & Payload Specialist: MDP Trajectory Logger, Empirical Return & Transition Matrix

* **Trajectory** $\tau=(s_0,a_0,r_0,s_1,a_1,r_1,\dots,s_T)$ is recorded by `TrajectoryLogger` (`start_episode`, `log_step`, `end_episode`).
* **Return** $G_0=\sum_t\gamma^t r_t$ with $\gamma=0.95$, evaluated backwards as $G_t=r_t+\gamma G_{t+1}$.
* **Transition frequencies** $\hat P(s'\mid s,a)=N(s,a,s')/N(s,a)$.

> **Data note.** The briefing mentions a *provided* historical dataset of past jumps. It is not in the course repository (the instructor repository has none either), so `JUMP_LOG` is a **mock** written by hand so every probability can be checked by counting. `load_jumps_csv` accepts the real file (`state,action,next_state`) when it exists.

In [4]:
log = mdp_mod.TrajectoryLogger(gamma=0.95)
log.start_episode("s0")
for action, reward, nxt in [("a0", 10, "s1"), ("a1", 0, "s2"), ("a2", -5, "s3"), ("a3", 20, "s4")]:
    log.log_step(action, reward, nxt)
tau = log.end_episode()
print("tau =", tau.as_sequence())
print(f"G_0 = {tau.discounted_return(0.95):.6f}   (10 + 0.95^2*(-5) + 0.95^3*20 = 22.635)")
print("G_t for every step:", [round(g, 4) for g in tau.returns_to_go(0.95)])

model = mdp_mod.TransitionModel.from_jumps(mdp_mod.JUMP_LOG)
print(f"\nEmpirical P(s'|s,a) from {model.total_jumps} jumps (MOCK log); each row sums to 1:")
print(model.format_table())

tau = ['s0', 'a0', 10.0, 's1', 'a1', 0.0, 's2', 'a2', -5.0, 's3', 'a3', 20.0, 's4']
G_0 = 22.635000   (10 + 0.95^2*(-5) + 0.95^3*20 = 22.635)
G_t for every step: [22.635, 13.3, 14.0, 20.0]

Empirical P(s'|s,a) from 56 jumps (MOCK log); each row sums to 1:
                                           dust_lane escape_corridor homunculus_core     hull_breach  outer_filament    plasma_shear      N
dust_lane | warp_high_shields                  0.000           0.600           0.300           0.100           0.000           0.000     10
dust_lane | warp_low_shields                   0.000           0.000           0.500           0.500           0.000           0.000      4
homunculus_core | hold_and_scan                0.000           0.000           0.667           0.333           0.000           0.000      3
homunculus_core | warp_high_shields            0.000           0.571           0.000           0.429           0.000           0.000      7
outer_filament | warp_high_shields          

The Part 3 file also holds the **loop sentinel** that Part 4 uses. Each step is hashed with SHA-256 over (state, action, arguments), so duplicate detection is an $O(1)$ dictionary lookup. The tripwire fires after three identical steps in a row, or when Floyd's tortoise and hare finds a cycle in the trajectory and the current state+action has been seen three times. The loop $s_1\to s_2\to s_3\to s_1$ shows why the second rule is needed: a consecutive-repeat counter alone never sees it.

In [5]:
seq = [("s1", "a1"), ("s2", "a2"), ("s3", "a3")] * 4
naive = sentinel_mod.InfiniteLoopSentinel()
print("consecutive-only check_loop() flags the loop over", len(seq), "steps:",
      any(naive.check_loop(a, {}, state=s)[0] for s, a in seq))

guard = sentinel_mod.InfiniteLoopSentinel(repeat_threshold=3)
for step, (state, action) in enumerate(seq, start=1):
    report = guard.observe(action, {}, state=state)
    if report.tripped:
        break
print(f"observe() tripped at step {step}: {report.reason}")
print("Floyd (cycle start index, length) =", report.cycle)

SECURITY VIOLATION: infinite loop detected at step 7 (cycle of length 3 starting at step 1; this state+action seen 3x): action=a1 state=s1


consecutive-only check_loop() flags the loop over 12 steps: False
observe() tripped at step 7: cycle of length 3 starting at step 1; this state+action seen 3x
Floyd (cycle start index, length) = (0, 3)


---
## Part 4 — Security & Ethics Sentinel: System 2 Warp Pilot & Emergency Tripwire

`WarpPilot` wires the four parts together. For every proposed action the System 2 engine audits it first (unknown tools are rejected, shield intensity below the safety margin of 60% is corrected, and warp navigation is blocked until shields are up and the escape corridor has been scanned). Approved actions go through the Part 2 dispatcher, each executed step is logged as an MDP transition (Part 3), the Part 1 state machine is stepped through its cycle, and the Part 3 sentinel watches every state+action. When it trips, `emergency_halt()` forces `HALTED` and logs a critical security violation.

**Scenario A — escape mission** (scan the corridor, raise shields, warp out):

In [6]:
logging.getLogger("warp_pilot").setLevel(logging.ERROR)
escape = pilot_mod.WarpPilot(agent_mod, protocol_mod, sentinel_mod, mdp_mod)
outcome = escape.run_escape_mission()

print()
for dec, obs in zip(outcome.decisions, outcome.observations):
    print(f"[{dec['status']:<8}] {dec['action']}({dec['arguments']})")
    print(f"           {dec['deliberation_trace']}")
    print(f"           {obs}")
traj = outcome.trajectory
print(f"\nfinal state={outcome.agent.state.name}  mission_completed={outcome.agent.mission_completed}  "
      f"steps={outcome.agent.step_count}/{outcome.agent.max_steps}  violations={outcome.violations}")
print(f"trajectory: T={traj.T}, rewards={list(traj.rewards)}, G_0={traj.discounted_return(0.95):.4f}")

Step 01: Transitioned from IDLE -> PERCEIVING
Step 02: Transitioned from PERCEIVING -> DELIBERATING
Step 03: Transitioned from DELIBERATING -> ACTING
Step 04: Transitioned from ACTING -> OBSERVING
Step 05: Transitioned from OBSERVING -> REFLECTING
Step 06: Transitioned from REFLECTING -> DELIBERATING
Step 07: Transitioned from DELIBERATING -> ACTING
Step 08: Transitioned from ACTING -> OBSERVING
Step 09: Transitioned from OBSERVING -> REFLECTING
Step 10: Transitioned from REFLECTING -> DELIBERATING
Step 11: Transitioned from DELIBERATING -> ACTING
Step 12: Transitioned from ACTING -> OBSERVING
Step 13: Transitioned from OBSERVING -> REFLECTING
Step 14: Transitioned from REFLECTING -> HALTED

[APPROVED] query_sensor_array({'target': 'escape_corridor'})
           System 1: Locate a safe exit | System 2: Passed safety audit
           Observation: Clear gravitational corridor detected along vector [-0.85, 0.52, 0.05].
[MODIFIED] adjust_plasma_shields({'intensity': 60})
           System 

**Scenario B — Ship Anomaly 06:** AURA repeats the flawed sequence $s_1\to s_2\to s_3\to s_1\dots$ and the circuit breaker must stop her.

In [7]:
anomaly = pilot_mod.WarpPilot(agent_mod, protocol_mod, sentinel_mod, mdp_mod)
outcome = anomaly.run_loop_anomaly()
print(f"final state={outcome.agent.state.name}  actions observed before halt={len(anomaly.sentinel.history_hashes)}")
for v in outcome.violations:
    print("violation:", v)

SECURITY VIOLATION: infinite loop detected at step 7 (cycle of length 3 starting at step 1; this state+action seen 3x): action=a1 state=s1


SECURITY VIOLATION: Emergency Tripwire at attempt 7: cycle of length 3 starting at step 1; this state+action seen 3x


final state=HALTED  actions observed before halt=7
violation: SECURITY VIOLATION: Emergency Tripwire at attempt 7: cycle of length 3 starting at step 1; this state+action seen 3x


---
## Master Test Suite

The four suites required by the briefing, written as plain `assert` statements against the loaded modules.

### Test 1 — State Machine Test

In [8]:
# (a) IDLE -> ... -> HALTED through the strict graph when the mission completes
a = Agent(verbose=False)
path = [a.state]
for _ in range(5):
    path.append(a.step())
a.mission_completed = True
path.append(a.step())
S = AgentState
assert path == [S.IDLE, S.PERCEIVING, S.DELIBERATING, S.ACTING, S.OBSERVING, S.REFLECTING, S.HALTED], path
assert a.state == S.HALTED and a.step_count == 6

# (b) a run that never completes is forced to stop: 20 transitions, then the 21st call halts it
b = Agent(max_steps=20, verbose=False)
for call in range(1, 21):
    b.step()
    assert b.state != S.HALTED, f"halted too early at call {call}"      # REFLECTING loops forever otherwise
assert b.step_count == 20
b.step()                                                                # call 21
assert b.state == S.HALTED and b.step_count == 20 and "budget" in b.halt_reason
b.step()                                                                # HALTED is terminal
assert b.state == S.HALTED and b.step_count == 20

# (c) the budget itself cannot exceed 20
for bad in (0, 21, 100):
    try:
        Agent(max_steps=bad)
        raise AssertionError(f"max_steps={bad} accepted")
    except ValueError:
        pass
print("TEST 1 PASSED: IDLE -> HALTED, strict termination at step 21, budget capped at 20")

TEST 1 PASSED: IDLE -> HALTED, strict termination at step 21, budget capped at 20


### Test 2 — Protocol Dispatch Test

In [9]:
dispatcher = protocol_mod.ActionDispatcher()
bad_payloads = ['{"intensity": 85', "not json at all", "", None, [85], 85, '[85]', '{"intensity": "high"}', {"level": 85}]
for payload in bad_payloads:
    out = dispatcher.dispatch("adjust_plasma_shields", payload)        # must not raise
    assert isinstance(out, str) and out.startswith("Tool Execution Error"), (payload, out)

out = dispatcher.dispatch("adjust_plasma_shields", {"intensity": -10})   # tool-level ValueError
assert out.startswith("Tool Execution Error") and "ValueError" in out

assert dispatcher.dispatch("adjust_plasma_shields", {"intensity": 85}).startswith("Observation:")
assert dispatcher.dispatch("adjust_plasma_shields", '{"intensity": 42}').startswith("Observation:")
print("TEST 2 PASSED: invalid payloads return formatted error strings; valid calls still execute")

TEST 2 PASSED: invalid payloads return formatted error strings; valid calls still execute


### Test 3 — MDP Return Test

In [10]:
rewards, gamma = [10, 0, -5, 20], 0.95
expected = 10 * gamma**0 + 0 * gamma**1 + (-5) * gamma**2 + 20 * gamma**3    # term by term
assert math.isclose(expected, 22.635, abs_tol=1e-12)
assert math.isclose(mdp_mod.discounted_return(rewards, gamma), expected, abs_tol=1e-12)

logger = mdp_mod.TrajectoryLogger(gamma=gamma)
logger.start_episode("s0")
for i, r in enumerate(rewards):
    logger.log_step(f"a{i}", r, f"s{i + 1}")
logger.end_episode()
assert math.isclose(logger.returns()[0], expected, abs_tol=1e-12)
print(f"TEST 3 PASSED: G_0 = {expected:.3f} for rewards {rewards} at gamma = {gamma}")

TEST 3 PASSED: G_0 = 22.635 for rewards [10, 0, -5, 20] at gamma = 0.95


### Test 4 — Sentinel Cycle Test

In [11]:
cycle_seq = [("s1", "a1"), ("s2", "a2"), ("s3", "a3")] * 4        # s1, s2, s3, s1, s2, s3, ...
sentinel = sentinel_mod.InfiniteLoopSentinel(repeat_threshold=3)
agent = Agent(max_steps=20, verbose=False)
halted_at, report = None, None
for i, (state, action) in enumerate(cycle_seq, start=1):
    agent.step()
    report = sentinel.observe(action, {}, state=state)
    if report.tripped:
        agent.halt(f"Emergency Tripwire: {report.reason}")          # circuit breaker -> HALTED
        halted_at = i
        break

hashes = [sentinel.compute_step_hash(a, {}, state=s) for s, a in cycle_seq[:halted_at]]
assert sentinel_mod.floyd_cycle(hashes) == (0, 3)                   # tortoise and hare found the 3-cycle
assert report.cycle == (0, 3)
assert halted_at is not None and halted_at < 10
assert agent.state == AgentState.HALTED and agent.step_count < 10
print(f"TEST 4 PASSED: cycle (start 0, length 3) detected, agent HALTED at step {halted_at} (< 10)")

SECURITY VIOLATION: infinite loop detected at step 7 (cycle of length 3 starting at step 1; this state+action seen 3x): action=a1 state=s1


TEST 4 PASSED: cycle (start 0, length 3) detected, agent HALTED at step 7 (< 10)


### Full per-part self-test suites

In [12]:
logging.getLogger("sentinel").setLevel(logging.CRITICAL + 1)
logging.getLogger("warp_pilot").setLevel(logging.CRITICAL + 1)
agent_mod.run_self_tests();                      print()
protocol_mod.run_protocol_dispatch_tests();      print()
mdp_mod.run_self_tests();                        print()
sentinel_mod.run_self_tests();                   print()
pilot_mod.run_part4_tests()


=== Mission 06 Part 1: State Machine Self-Tests ===
[PASS] AgentState has exactly the 7 required states
[PASS] Graph transitions in strict order, REFLECTING loops to DELIBERATING
[PASS] IDLE -> HALTED when mission_completed at REFLECTING
[PASS] step() performs exactly one transition per call
[PASS] HALTED is terminal (step() is a no-op)
[PASS] Step budget: 20 transitions, 21st call forces HALTED
[PASS] Custom budget (max_steps=3) enforced
[PASS] max_steps validated (1..20, int only)
[PASS] perception_input recorded; no shared mutable defaults
[PASS] External forced HALTED (tripwire) is respected

All Part 1 self-tests passed.

[PASS] All 19 Protocol Dispatch tests passed.

[TEST 1] MDP Return Test (briefing): rewards [10, 0, -5, 20], gamma = 0.95
         G_0 = 22.635000  (expected 22.635000)  PASS
[TEST 2] discounted_return edge cases and returns_to_go
         PASS
[TEST 3] TrajectoryLogger records tau = (s0, a0, r0, ..., sT)
         PASS
[TEST 4] TransitionModel on the mock jump l

---
### Notes

* **Budget semantics.** $N$ transitions are allowed; the $(N{+}1)$-th call to `step()` halts the agent without counting as a transition, so `step_count` never exceeds $N$. A forced `HALTED` records a `halt_reason` (`step budget of N exceeded`, `mission completed`, or the tripwire text).
* **Changes to the Part 1 starter.** The `if/elif` chain became a `TRANSITIONS` table; `HALTED` is now terminal (the starter counted a bogus "HALTED → HALTED" step); `max_steps` is validated; `perception_input` is stored instead of ignored. The public interface used by Parts 3 and 4 (`Agent(max_steps=...)`, `step()`, `state`, `step_count`, `mission_completed`) is unchanged, and Part 4 may still set `agent.state = HALTED` directly for the emergency tripwire.
* **Mock data.** The jump log in Part 3 and the demo rewards are illustrative; no claim is made about a real nebula dataset. Empirical probabilities are maximum-likelihood estimates from 56 jumps, so rows with few observations are noisy.
* **Tripwire limits.** The Floyd branch assumes a deterministic loop (the same state+action is always followed by the same next one); otherwise only the three-in-a-row rule applies.
* **AI assistance.** Claude helped complete the starters and write the self-checks for Parts 1 and 3; the author-review lines in those file headers still need to be updated by each officer once they have reviewed the code.